## Etapa 4: Exportação Power BI

In [ ]:
# ===================================================================
# SPOTIFY AMÉRICAS - EXPORTAÇÃO PARA POWER BI
# ===================================================================
# Autor: Seu Nome
# Data: 2024
# Objetivo: Exportar dados tratados para criação de dashboard no Power BI
# ===================================================================

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║   📊 SPOTIFY AMÉRICAS - EXPORTAÇÃO PARA POWER BI 📊                          ║
║                                                                              ║
║   Preparação dos dados para dashboard interativo no Power BI                 ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

# ===================================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ===================================================================

import pandas as pd
import numpy as np
import warnings
import os
import json

warnings.filterwarnings('ignore')

print("✅ Bibliotecas importadas com sucesso!")

# ===================================================================
# 2. CARREGAR DADOS LIMPOS
# ===================================================================

print("\n📂 CARREGANDO DADOS LIMPOS...")
print("=" * 60)

try:
    df = pd.read_parquet('/content/drive/MyDrive/spotify_americas_data/spotify_americas_clean.parquet')
    print("✅ Dados carregados com sucesso!")
except:
    df = pd.read_csv('/content/drive/MyDrive/spotify_americas_data/spotify_americas_clean.csv')
    print("✅ Dados carregados do CSV!")

print(f"📊 Shape: {df.shape[0]:,} registros")

# Criar pasta de exportação
output_dir = '/content/drive/MyDrive/spotify_americas_powerbi/'
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Pasta de exportação: {output_dir}")

# ===================================================================
# 3. TABELA 1: FATOS (DADOS COMPLETOS)
# ===================================================================

print("\n📊 CRIANDO TABELA FATO...")
print("-" * 40)

# Selecionar colunas para o Power BI
colunas_fato = [
    'track_name', 'artist_names', 'country', 'sub_region' if 'sub_region' in df.columns else 'region',
    'rank', 'streams_millions', 'week', 'year', 'month', 'month_name',
    'danceability', 'energy', 'valence', 'success_score',
    'music_profile' if 'music_profile' in df.columns else 'rank_group'
]

colunas_exist = [c for c in colunas_fato if c in df.columns]
df_fato = df[colunas_exist].copy()

# Renomear para português
rename_map = {
    'track_name': 'musica',
    'artist_names': 'artista',
    'country': 'pais',
    'sub_region': 'sub_regiao',
    'streams_millions': 'streams_milhoes',
    'week': 'data',
    'danceability': 'dancabilidade',
    'success_score': 'score_sucesso',
    'music_profile': 'perfil_musical'
}

df_fato = df_fato.rename(columns={k: v for k, v in rename_map.items() if k in df_fato.columns})

df_fato.to_csv(f'{output_dir}01_tabela_fato_musicas.csv', index=False)
print(f"✅ 01_tabela_fato_musicas.csv - {len(df_fato):,} registros")

# ===================================================================
# 4. TABELA 2: DIMENSÃO PAÍS
# ===================================================================

print("\n📊 CRIANDO TABELA DIMENSÃO PAÍS...")
print("-" * 40)

pais_agg = df.groupby('country').agg({
    'streams_millions': ['sum', 'mean', 'std'],
    'danceability': 'mean',
    'energy': 'mean',
    'valence': 'mean',
    'rank': 'mean',
    'track_name': 'count',
    'artist_names': 'nunique'
}).round(3)

pais_agg.columns = ['total_streams_m', 'media_streams_m', 'std_streams_m',
                    'media_dancabilidade', 'media_energia', 'media_valence',
                    'rank_medio', 'total_musicas', 'total_artistas']
pais_agg = pais_agg.reset_index()

# Adicionar sub-região
if 'sub_region' in df.columns:
    sub_map = df.groupby('country')['sub_region'].first()
    pais_agg['sub_regiao'] = pais_agg['country'].map(sub_map)

pais_agg.to_csv(f'{output_dir}02_dimensao_paises.csv', index=False)
print(f"✅ 02_dimensao_paises.csv - {len(pais_agg)} países")

# ===================================================================
# 5. TABELA 3: TOP MÚSICAS
# ===================================================================

print("\n📊 CRIANDO TABELA TOP MÚSICAS...")
print("-" * 40)

top_musicas = (df.groupby(['track_name', 'artist_names'])
               .agg({
                   'streams_millions': 'sum',
                   'rank': 'min',
                   'country': 'nunique'
               })
               .rename(columns={'country': 'paises_alcancados'})
               .sort_values('streams_millions', ascending=False)
               .head(100)
               .reset_index())

top_musicas.to_csv(f'{output_dir}03_top_100_musicas.csv', index=False)
print(f"✅ 03_top_100_musicas.csv - {len(top_musicas)} músicas")

# ===================================================================
# 6. TABELA 4: TOP ARTISTAS
# ===================================================================

print("\n📊 CRIANDO TABELA TOP ARTISTAS...")
print("-" * 40)

top_artistas = (df.groupby('artist_names')
                .agg({
                    'streams_millions': 'sum',
                    'track_name': 'nunique',
                    'country': 'nunique'
                })
                .rename(columns={'track_name': 'musicas_distintas', 'country': 'paises_alcancados'})
                .sort_values('streams_millions', ascending=False)
                .head(50)
                .reset_index())

top_artistas.to_csv(f'{output_dir}04_top_50_artistas.csv', index=False)
print(f"✅ 04_top_50_artistas.csv - {len(top_artistas)} artistas")

# ===================================================================
# 7. TABELA 5: SÉRIE TEMPORAL
# ===================================================================

print("\n📊 CRIANDO TABELA SÉRIE TEMPORAL...")
print("-" * 40)

temporal_semanal = df.groupby(['week', 'year', 'month']).agg({
    'streams_millions': 'sum',
    'rank': 'mean'
}).reset_index()

temporal_semanal.to_csv(f'{output_dir}05_serie_temporal.csv', index=False)
print(f"✅ 05_serie_temporal.csv - {len(temporal_semanal)} semanas")

# ===================================================================
# 8. TABELA 6: PERFIL MUSICAL
# ===================================================================

if 'music_profile' in df.columns:
    print("\n📊 CRIANDO TABELA PERFIL MUSICAL...")
    print("-" * 40)

    perfil_agg = df.groupby('music_profile').agg({
        'streams_millions': ['sum', 'mean'],
        'track_name': 'count'
    }).round(2)

    perfil_agg.columns = ['total_streams_m', 'media_streams_m', 'total_musicas']
    perfil_agg = perfil_agg.reset_index()
    perfil_agg.columns = ['perfil_musical', 'total_streams_m', 'media_streams_m', 'total_musicas']

    perfil_agg.to_csv(f'{output_dir}06_perfil_musical.csv', index=False)
    print(f"✅ 06_perfil_musical.csv - {len(perfil_agg)} perfis")

# ===================================================================
# 9. TABELA 7: ANÁLISE POR SUB-REGIÃO
# ===================================================================

if 'sub_region' in df.columns:
    print("\n📊 CRIANDO TABELA ANÁLISE POR SUB-REGIÃO...")
    print("-" * 40)

    regiao_agg = df.groupby('sub_region').agg({
        'streams_millions': 'sum',
        'danceability': 'mean',
        'energy': 'mean',
        'valence': 'mean',
        'country': 'nunique',
        'track_name': 'count'
    }).round(3).reset_index()

    regiao_agg.columns = ['sub_regiao', 'total_streams_m', 'media_dancabilidade',
                          'media_energia', 'media_valence', 'total_paises', 'total_musicas']

    regiao_agg.to_csv(f'{output_dir}07_analise_sub_regioes.csv', index=False)
    print(f"✅ 07_analise_sub_regioes.csv - {len(regiao_agg)} regiões")

# ===================================================================
# 10. METADADOS
# ===================================================================

print("\n📊 CRIANDO METADADOS...")
print("-" * 40)

metadados = {
    'projeto': 'Spotify Weekly Top 200 - Análise das Américas',
    'data_exportacao': pd.Timestamp.now().isoformat(),
    'total_registros': len(df),
    'total_paises': int(df['country'].nunique()),
    'total_musicas': int(df['track_name'].nunique()),
    'total_artistas': int(df['artist_names'].nunique()),
    'periodo_inicio': df['week'].min().isoformat(),
    'periodo_fim': df['week'].max().isoformat(),
    'total_streams_milhoes': float(df['streams_millions'].sum()),
    'media_dancabilidade': float(df['danceability'].mean()),
    'media_energia': float(df['energy'].mean()),
    'media_valence': float(df['valence'].mean()),
    'arquivos_exportados': [
        '01_tabela_fato_musicas.csv',
        '02_dimensao_paises.csv',
        '03_top_100_musicas.csv',
        '04_top_50_artistas.csv',
        '05_serie_temporal.csv',
        '06_perfil_musical.csv',
        '07_analise_sub_regioes.csv'
    ]
}

with open(f'{output_dir}metadados.json', 'w') as f:
    json.dump(metadados, f, indent=2, ensure_ascii=False)

print(f"✅ metadados.json salvo")

# ===================================================================
# 11. RESUMO DA EXPORTAÇÃO
# ===================================================================

print("\n" + "=" * 70)
print("✅ EXPORTAÇÃO PARA POWER BI CONCLUÍDA!")
print("=" * 70)

print(f"""
📁 ARQUIVOS GERADOS:

   📄 01_tabela_fato_musicas.csv - Dados completos ({len(df_fato):,} registros)
   📄 02_dimensao_paises.csv - Métricas por país ({len(pais_agg)} países)
   📄 03_top_100_musicas.csv - Top 100 músicas
   📄 04_top_50_artistas.csv - Top 50 artistas
   📄 05_serie_temporal.csv - Evolução temporal
   📄 06_perfil_musical.csv - Análise de perfis
   📄 07_analise_sub_regioes.csv - Análise por sub-região
   📄 metadados.json - Documentação

📂 LOCALIZAÇÃO: {output_dir}

🎯 PRÓXIMOS PASSOS NO POWER BI:
   1. Abra o Power BI Desktop
   2. Clique em "Obter Dados" → "Texto/CSV"
   3. Conecte cada arquivo CSV
   4. Crie relacionamentos entre as tabelas
   5. Construa as visualizações
""")

print("\n🎉 Exportação concluída com sucesso!")
print("=" * 70)